In [ ]:
from CLEAN.utils import *
dirs = "../local_snapshot/tttianhao-CLEAN-0cf2cac/"
#ensure_dirs(dirs+"data/esm_data")
#ensure_dirs(dirs+"data/pretrained")
#csv_to_fasta(dirs+"data/split50.csv", dirs+"data/split50.fasta")
#csv_to_fasta(dirs+"data/new.csv", dirs+"data/new.fasta")

In [ ]:
import math
def mutate(seq: str, position: int) -> str:
    seql = seq[ : position]
    seqr = seq[position+1 : ]
    seq = seql + '*' + seqr
    return seq

def mask_sequences(single_id, csv_name, fasta_name) :
    csv_file = open(dirs + 'data/'+ csv_name + '.csv')
    csvreader = csv.reader(csv_file, delimiter = '\t')
    output_fasta = open(dirs + 'data/' + fasta_name + '.fasta','w')
    single_id = set(single_id)
    for i, rows in enumerate(csvreader):
        if rows[0] in single_id:
            for j in range(10):
                seq = rows[2].strip()
                mu, sigma = .10, .02 # mean and standard deviation
                s = np.random.normal(mu, sigma, 1)
                mut_rate = s[0]
                times = math.ceil(len(seq) * mut_rate)
                for k in range(times):
                    position = random.randint(1 , len(seq) - 1)
                    seq = mutate(seq, position)
                seq = seq.replace('*', '<mask>')
                output_fasta.write('>' + rows[0] + '_' + str(j) + '\n')
                output_fasta.write(seq + '\n')

def mutate_single_seq_ECs(train_file):
    id_ec, ec_id =  get_ec_id_dict(dirs + 'data/' + train_file + '.csv')
    single_ec = set()
    for ec in ec_id.keys():
        if len(ec_id[ec]) == 1:
            single_ec.add(ec)
    single_id = set()
    for id in id_ec.keys():
        for ec in id_ec[id]:
            if ec in single_ec:
                single_id.add(id)
                break
    print("Number of EC numbers with only one sequences:",len(single_ec))
    print("Number of single-seq EC number sequences need to mutate: ",len(single_id))
    print("Number of single-seq EC numbers already mutated: ", len(single_ec) - len(single_id))
    mask_sequences(single_id, train_file, train_file+'_single_seq_ECs')
    fasta_name = train_file+'_single_seq_ECs'
    return fasta_name


In [ ]:


train_file = "split50"

train_fasta_file = mutate_single_seq_ECs(train_file)



In [ ]:
id_ec, ec_id =  get_ec_id_dict(dirs+'data/split10_re2' + '.csv')

In [ ]:
length_key = []
for key in ec_id.keys():
    length_key.append(len(ec_id[key]))

In [ ]:
np.sort(length_key)[:1500]

In [ ]:
ensure_dirs(dirs+'/data/distance_map/')
id_ec_dict, ec_id_dict = get_ec_id_dict(dirs+"data/split50.csv")
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
dtype = torch.float32

In [ ]:
file = '../local_snapshot/data/Embedding/sep1_s_1024_mlp_aug_dic.h5'

import h5py
import torch
def load_prost5_o(lookup):
    if len(lookup.split('_')) == 1:
        if lookup in id_ec_dict50.keys():
            with h5py.File(file, 'r') as hf:
                dataset_name = lookup 
                dataset = hf[dataset_name]  

                data = dataset[:]
        elif lookup in id_ec_test.keys():
            with h5py.File(file3, 'r') as hf:
                dataset_name = lookup 
                dataset = hf[dataset_name]  

                data = dataset[:]
             
        else:
            with h5py.File(file2, 'r') as hf:
                dataset_name = lookup 
                dataset = hf[dataset_name]  

                data = dataset[:]
        
    else:
        with h5py.File(file1, 'r') as hf:
            dataset_name = lookup 
            dataset = hf[dataset_name]  

            data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)


def load_prost5(lookup):
    
    with h5py.File(file, 'r') as hf:
        dataset_name = lookup 
        dataset = hf[dataset_name]  

        data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)[0]


def pro_embedding(ec_id_dict, device, dtype):
    '''
    Loading esm embedding in the sequence of EC numbers
    prepare for calculating cluster center by EC
    '''
    pro_emb = []
    # for ec in tqdm(list(ec_id_dict.keys())):
    for ec in list(ec_id_dict.keys()):
        ids_for_query = list(ec_id_dict[ec])
        pro_to_cat = [load_prost5(id) for id in ids_for_query]
        pro_emb = pro_emb + pro_to_cat
    return torch.cat(pro_emb).to(device=device, dtype=dtype)

prost5_emb = pro_embedding(ec_id_dict, device, dtype)
#esm_dist = get_dist_map(ec_id_dict, esm_emb, device, dtype)




In [ ]:
prost5_emb.shape

In [ ]:
load_prost5('Q08890')

In [ ]:
pro_dist = get_dist_map(ec_id_dict, prost5_emb, device, dtype)

In [ ]:
pickle.dump(pro_dist, open(dirs+'/data/distance_map/' + 'split50' + '.pkl', 'wb'))
pickle.dump(prost5_emb, open(dirs+'/data/distance_map/' + 'split50' + '_esm.pkl', 'wb'))

In [ ]:
!python3 ../local_snapshot/tttianhao-CLEAN-0cf2cac/train-triplet.py --training_data split50 --model_name sep1_s_1024_mlp --epoch 3500

In [ ]:
import torch
import torch.nn as nn
class LayerNormNet(nn.Module):
    def __init__(self, hidden_dim, out_dim, device, dtype, drop_out=0.1):
        super(LayerNormNet, self).__init__()
        self.hidden_dim1 = hidden_dim
        self.out_dim = out_dim
        self.drop_out = drop_out
        self.device = device
        self.dtype = dtype

        self.fc1 = nn.Linear(1024, hidden_dim, dtype=dtype, device=device)
        self.ln1 = nn.LayerNorm(hidden_dim, dtype=dtype, device=device)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim,
                             dtype=dtype, device=device)
        self.ln2 = nn.LayerNorm(hidden_dim, dtype=dtype, device=device)
        self.fc3 = nn.Linear(hidden_dim, out_dim, dtype=dtype, device=device)
        self.dropout = nn.Dropout(p=drop_out)

    def forward(self, x):
        

        x = self.dropout(self.ln1(self.fc1(x)))
        x = torch.relu(x)
        x = self.dropout(self.ln2(self.fc2(x)))
        x = torch.relu(x)
        x = self.fc3(x)
        return x
 


In [ ]:
checkpoint = torch.load(dirs+'/data/model/'+ 'split50_triplet' +'.pth')

In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
dtype = torch.float32
model = LayerNormNet(512, 128, device, dtype)
model.load_state_dict(checkpoint)

In [ ]:
id_ec_test, ec_id_test =  get_ec_id_dict(dirs+'data/new' + '.csv')
id_ec_test_fli = {key: id_ec_test[key] for key in test_pre.keys() if key in id_ec_test}

In [ ]:
def csv_to_csv(csv_name,fasta_name,id_ec):
    csvfile = open(csv_name, 'r')
    csvreader = csv.reader(csvfile, delimiter='\t')
    outfile = open(fasta_name, 'w')
    status = False
    
    for i, rows in enumerate(csvreader):
        if i > 0 and rows[0] in test_pre.keys():
            ec = id_ec[rows[0]]
            
            outfile.write(rows[0]+'\t')
            for i in range(len(ec)):
                if len(ec)-i > 1:
                    outfile.write(str(ec[i])+';')
                else:

                    outfile.write(str(ec[i])+'\t')
                print(rows[0])
                print('1:'+'\t'+str(ec[i])+'\t')
            outfile.write(rows[2]+'\n')

ori_test='../local_snapshot/tttianhao-CLEAN-0cf2cac/data/new.csv'            
new_test = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/new_test.csv'          
csv_to_csv(ori_test,new_test,id_ec_test_fli)

In [ ]:
#from CLEAN.model import LayerNormNet
file = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/split50_seq.h5'
file1 = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/split50_single_seq.h5'
file2 = '../local_snapshot/tttianhao-CLEAN-0cf2cac/data/prost5/new_seq.h5'

import h5py
import torch
def load_prost5(lookup):
    if len(lookup.split('_')) == 1:
        if lookup in id_ec_test.keys():
            with h5py.File(file2, 'r') as hf:
                dataset_name = lookup 
                dataset = hf[dataset_name]  

                data = dataset[:]
        else:   
            with h5py.File(file, 'r') as hf:
                dataset_name = lookup 
                dataset = hf[dataset_name]  

                data = dataset[:]
    else:
        with h5py.File(file1, 'r') as hf:
            dataset_name = lookup 
            dataset = hf[dataset_name]  

            data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)

def pro_embedding(ec_id_dict, device, dtype):
    '''
    Loading esm embedding in the sequence of EC numbers
    prepare for calculating cluster center by EC
    '''
    pro_emb = []
    # for ec in tqdm(list(ec_id_dict.keys())):
    for ec in list(ec_id_dict.keys()):
        ids_for_query = list(ec_id_dict[ec])
        pro_to_cat = [load_prost5(id) for id in ids_for_query]
        pro_emb = pro_emb + pro_to_cat
    return torch.cat(pro_emb).to(device=device, dtype=dtype)


def model_embedding_test(id_ec_test, model, device, dtype):
    '''
    Instead of loading esm embedding in the sequence of EC numbers
    the test embedding is loaded in the sequence of queries
    then inferenced with model to get model embedding
    '''
    ids_for_query = list(id_ec_test.keys())
    pro_to_cat = [load_prost5(id) for id in ids_for_query]
    pro_emb = torch.cat(pro_to_cat).to(device=device, dtype=dtype)
    model_emb = model(pro_emb)
    return model_emb


def get_eval_metrics(pred_label, pred_probs, true_label, all_label):
    mlb = MultiLabelBinarizer()
    mlb.fit([list(all_label)])
    n_test = len(pred_label)
    pred_m = np.zeros((n_test, len(mlb.classes_)))
    true_m = np.zeros((n_test, len(mlb.classes_)))
    # for including probability
    pred_m_auc = np.zeros((n_test, len(mlb.classes_)))
    label_pos_dict = get_ec_pos_dict(mlb, true_label, pred_label)
    for i in range(n_test):
        pred_m[i] = mlb.transform([pred_label[i]])
        true_m[i] = mlb.transform([true_label[i]])
         # fill in probabilities for prediction
        labels, probs = pred_label[i], pred_probs[i]
        for label, prob in zip(labels, probs):
            if label in all_label:
                pos = label_pos_dict[label]
                pred_m_auc[i, pos] = prob
    pre = precision_score(true_m, pred_m, average='weighted', zero_division=0)
    rec = recall_score(true_m, pred_m, average='weighted')
    f1 = f1_score(true_m, pred_m, average='weighted')
    #roc = roc_auc_score(true_m, pred_m_auc, average='weighted')
    acc = accuracy_score(true_m, pred_m)
    return pre, rec, f1,  acc



def infer_maxsep(train_data, test_data, report_metrics = False, 
                 pretrained=True, model_name=None):
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda:0" if use_cuda else "cpu")
    dtype = torch.float32
    id_ec_train, ec_id_dict_train = get_ec_id_dict(dirs+'/data/' + train_data + '.csv')
    id_ec_test, _ = get_ec_id_dict(dirs+'/data/' + test_data + '.csv')
    
    # load checkpoints
    # NOTE: change this to LayerNormNet(512, 256, device, dtype) 
    # and rebuild with [python build.py install]
    # if inferencing on model trained with supconH loss
    model = LayerNormNet(512, 128, device, dtype)
    
    if pretrained:
        try:
            checkpoint = torch.load(dirs+'/data/pretrained/'+ train_data +'.pth')
        except FileNotFoundError as error:
            raise Exception('No pretrained weights for this training data')
    else:
        try:
            checkpoint = torch.load(dirs+'/data/model/'+ model_name +'.pth')
        except FileNotFoundError as error:
            raise Exception('No model found!')
            
    model.load_state_dict(checkpoint)
    model.eval()
    # load precomputed EC cluster center embeddings if possible
    if train_data == "split70":
        emb_train = torch.load(dirs+'/data/pretrained/70.pt')
    elif train_data == "split100":
        emb_train = torch.load(dirs+'/data/pretrained/100.pt')
    else:
        emb_train = model(pro_embedding(ec_id_dict_train, device, dtype))
        
    emb_test = model_embedding_test(id_ec_test, model, device, dtype)
    eval_dist = get_dist_map_test(emb_train, emb_test, ec_id_dict_train, id_ec_test, device, dtype)
    seed_everything()
    eval_df = pd.DataFrame.from_dict(eval_dist)
    ensure_dirs(dirs+"/results")
    out_filename = dirs+"results/" +  test_data
    write_max_sep_choices(eval_df, out_filename)
    if report_metrics:
        pred_label = get_pred_labels(out_filename, pred_type='_maxsep')
        pred_probs = get_pred_probs(out_filename, pred_type='_maxsep')
        true_label, all_label = get_true_labels(dirs+'/data/' + test_data)
        pre, rec, f1, acc = get_eval_metrics(
            pred_label, pred_probs, true_label, all_label)
        print("############ EC calling results using maximum separation ############")
        print('-' * 75)
        print(f'>>> total samples: {len(true_label)} | total ec: {len(all_label)} \n'
            f'>>> precision: {pre:.3} | recall: {rec:.3}'
            f'| F1: {f1:.3}| ACC: {acc:.3}  ')
        print('-' * 75)


In [ ]:
infer_maxsep("split50", "new_test", report_metrics=True,
pretrained=False, model_name="split50_triplet")

In [ ]:
test_file = open('../local_snapshot/data/new.txt','r')
test_data = []
for line in test_file:
    test_data.append(line.split('\n')[0])
    

In [ ]:
file1 = '../local_snapshot/data/uniprotkb_AND_reviewed_true_2023_07_03.tsv'
file3 = '../local_snapshot/data/all/all_cluster.tsv'
file4 = '../local_snapshot/data/all/afall_ID50_cluster.tsv'
file5 = '../local_snapshot/data/all/select_cluster.tsv'
from CLEAN.utils import *
dirs = "../local_snapshot/tttianhao-CLEAN-0cf2cac/"
id_ec, ec_id =  get_ec_id_dict(dirs+'data/split100' + '.csv')

In [ ]:
rep_seq = {}
rep_fold = {}
seqfile = open(file3,'r')
for line in seqfile:
    key = line.split('\t')[0]
    if key in rep_seq.keys():
        rep_seq[key].append(line.split('\t')[1].split('\n')[0])
    else:
        rep_seq[key] = []
        rep_seq[key].append(line.split('\t')[1].split('\n')[0])
        
foldfile = open(file5,'r')
for line in foldfile:
    key = line.split('\t')[0].split('.')[0]
    if key in rep_fold.keys():
        rep_fold[key].append(line.split('\t')[1].split('.')[0])
    else:
        rep_fold[key] = []
        rep_fold[key].append(line.split('\t')[1].split('.')[0])
import os
import numpy as np
def get_pLDDT(pdb_p):
    infile = open(pdb_p,'r')
    pLDDT = []
    for line in infile:
          if line[0:6].rstrip() == 'ATOM' and line[13:15].rstrip() == 'CA':
            pLDDT.append(float(line[60:65].rstrip()))
    infile.close()
    return np.mean(pLDDT)
af_path = '../local_snapshot/data/afall/'
af_list = os.listdir(af_path)
rep_structure = []
rep_pLDDT = []
keys = []
for key in rep_seq.keys():
    pLDDTs = []
    items = []
    for item in rep_seq[key]:
        item_name = item + '.pdb'
        if item_name in af_list:
            pdb_p = af_path+ item + '.pdb'
            pLDDTs.append(get_pLDDT(pdb_p))
            items.append(item)
    if len(pLDDTs)>0:
        max_plddt = max(pLDDTs)
        rep_pLDDT.append(max_plddt)
        rep_structure.append(items[pLDDTs.index(max_plddt)])
        keys.append(key)
        
threshold_value = 90 
positions = [index for index, value in enumerate(rep_pLDDT) if value > threshold_value]

sel_struc=[rep_structure[index ]for index in positions]
sel_keys = [keys[index ]for index in positions]

fun_fold = {}
sort_items = []
sel_seq = {}
for key in rep_fold.keys():
    fun_fold[key] = []
    
    for item in rep_fold[key]:
        if item in sel_struc:
            k = sel_keys[sel_struc.index(item)]
            fun_fold[key].extend(rep_seq[k])
            sel_seq[k] = rep_seq[k]
        
            sort_items.extend(rep_seq[k])
        #print(key,item,len(sort_items))
        if item not in sel_struc and len(rep_fold[key])==1:
            fun_fold[key].append(key)
            sort_items.append(key)


In [ ]:
fun_check = {}
fun_key = {}
fold_index = []
fold_items = []
for key in fun_fold.keys():
    fun_key[key] = id_ec[key]
    fun_check[key]= []
    for item in fun_fold[key]:
        fun_check[key].append(id_ec[item])
        fold_index.append(key)
        fold_items.append(item)

fun_clu = {}
for key in ec_id.keys():
    fun_clu[key] = []
    for item in ec_id[key]:
        if item in fold_items:
            fun_clu[key].append(fold_index[fold_items.index(item)])
    if len(fun_clu[key]) == 0:
        fun_clu.pop(key, None)


uni_fold = []
for key in fun_clu.keys():
    list_of_lists=fun_clu[key]
    uni_fold.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))
    
uni_fun = []
for key in fun_check.keys():
    list_of_lists=fun_check[key]
    uni_fun.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))
    



In [ ]:
fun_check_fold = {}
fun_key_fold = {}
fold_index_fold = []
fold_items_fold = []
for key in rep_fold.keys():
    fun_key_fold[key] = id_ec[key]
    fun_check_fold[key]= []
    for item in rep_fold[key]:
        fun_check_fold[key].append(id_ec[item])
        fold_index_fold.append(key)
        fold_items_fold.append(item)

uni_fun_fold = []
for key in fun_check_fold.keys():
    list_of_lists=fun_check_fold[key]
    uni_fun_fold.append(all(sublist == list_of_lists[0] for sublist in list_of_lists))

sel_seq = {}
for item in sel_struc:
    k = sel_keys[sel_struc.index(item)]
    sel_seq[k] = rep_seq[k]
    
members = []
keys = []
for key in sel_seq.keys():
    keys.append(key)
    members.append(len(sel_seq[key]))
    
sing_seq = [keys[index] for index, value in enumerate(members) if value == 1]

members = []
keys = []
for key in rep_fold.keys():
    keys.append(key)
    members.append(len(rep_fold[key]))
sing_fold = [keys[index] for index, value in enumerate(members) if value == 1]

members = []
keys = []
for key in fun_fold.keys():
    keys.append(key)
    members.append(len(fun_fold[key]))
sing_funfold = [keys[index] for index, value in enumerate(members) if value == 1]

members = []
keys = []
for key in fun_clu.keys():
    keys.append(key)
    members.append(len(fun_clu[key]))
sing_fold_fun = [keys[index] for index, value in enumerate(members) if value == 1]

keys = []
for key in fun_check_fold.keys():
    keys.append(key)
uni_fun_fold_keys = []
uni_nofun_fold_keys = []
for i in range(len(keys)):
    if uni_fun[i]:
        uni_fun_fold_keys.append(keys[i])
    else:
        uni_nofun_fold_keys.append(keys[i])

In [ ]:
file_pre = '../local_snapshot/data/all/new_foldseek_key_cluster.tsv'
rep_pre = {}

seqfile = open(file_pre,'r')
for line in seqfile:
    key = line.split('\t')[0].split('.')[0]
    if key in rep_pre.keys():
        rep_pre[key].append(line.split('\t')[1].split('.')[0])
    else:
        rep_pre[key] = []
        rep_pre[key].append(line.split('\t')[1].split('.')[0])
        


In [ ]:
keys = []
for key in fun_fold.keys():
    keys.append(key)
keys_pre = []
items_pre = []
for key in rep_pre.keys():
    for item in rep_pre[key]:
        keys_pre.append(key)
        items_pre.append(item)

fun_check_pre = {}
for item in items_pre :
    if item in keys and item in uni_nofun_fold_keys:
         fun_check_pre[keys_pre[items_pre.index(item)]]=id_ec[item]
        
test_file = open('../local_snapshot/data/new.txt','r')
test_data = []
for line in test_file:
    test_data.append(line.split('\n')[0])
    
test_pre = {}
y_pre = []
test_true = {}
y_true = []
for item in test_data:
    if item in items_pre:
        if keys_pre[items_pre.index(item)] in fun_check_pre.keys():
            test_pre[item] = fun_check_pre[keys_pre[items_pre.index(item)]]
            y_pre.append(fun_check_pre[keys_pre[items_pre.index(item)]])
            test_true[item] = id_ec_test[item]
            y_true.append(id_ec_test[item])

In [ ]:
# 原始列表
versions = ['4.2.3.115', '4.2.3.114', '4.2.3.15', '4.2.3.113']

# 定义一个自定义的排序函数，按照版本号的逻辑进行排序
def custom_sort(version):
    return tuple(map(int, version.split('.')))

# 对列表进行排序，并使用自定义的排序函数
sorted_versions = sorted(versions, key=custom_sort)

# 打印排序后的结果
print("Sorted Versions:", str(sorted_versions))


In [ ]:
true_labels_full

In [ ]:
id_ec_test, ec_id_test =  get_ec_id_dict(dirs+'data/new' + '.csv')
Pos_pre = []
true_labels = []
pred_labels = []
true_labels_full = []
pred_labels_full = []

for key in test_pre.keys():
    true_labels_full.append(str(sorted(test_true[key], key=custom_sort)))
    true_labels.append(1)
    pred_labels_full.append(str(sorted(test_pre[key], key=custom_sort)))
    if test_pre[key] == test_true[key]:
        Pos_pre.append(key)
        pred_labels.append(1)
    else:
        pred_labels.append(0)

In [ ]:
true_labels_rou = []
pred_labels_rou = []

for key in test_pre.keys():
    
    true_labels_rou.append(1)
   
    if len(set(test_pre[key]).intersection(set(test_true[key]))) > 0:
        Pos_pre.append(key)
        pred_labels_rou.append(1)
    else:
        pred_labels_rou.append(0)

In [ ]:
len(pred_labels_rou)

In [ ]:
len(true_labels_rou)

In [ ]:
set(list1).intersection(set(list2))

In [ ]:
CLEAN: F1: 0.392 ACC: 0.481 
Foldseek_exact: F1: 0.347 ACC: 0.210           
Foldseek_rough: F1: 0.387 ACC: 0.240

In [ ]:
consistent
CLEAN: F1: 0.871  ACC: 0.857 
Foldseek_exact: F1: 0.643 ACC: 0.474        
Foldseek_rough: F1: 0.674 ACC: 0.509

In [ ]:
non
CLEAN:F1: 0.311 ACC: 0.391 
Foldseek_exact: F1: 0.232 ACC: 0.131
Foldseek_rough: F1: 0.273 ACC: 0.158    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data
metrics = ['F1', 'ACC']
methods = ['CLEAN', 'Foldseek_exact', 'Foldseek_rough']
scores = {'CLEAN': [0.311, 0.391],
          'Foldseek_exact': [0.273, 0.158],
          'Foldseek_rough': [0.232, 0.131]}

# Set the width of the bars
bar_width = 0.3

# Set the x locations for the groups
index = np.arange(len(metrics))

# Plot
plt.figure(figsize=(10, 6))

for i, method in enumerate(methods):
    plt.bar(index + (i - 1) * bar_width, scores[method], bar_width, label=method)

plt.xlabel('Metrics')
plt.ylabel('Scores')
plt.title('Performance Comparison in test data of EC not consistent cluster')
plt.xticks(index, metrics)
plt.legend()
plt.grid(False)
plt.show()



In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score


f1 = f1_score(true_labels_rou, pred_labels_rou, average='weighted')
# 计算精确度
precision = precision_score(true_labels_rou, pred_labels_rou)

# 计算召回率
recall = recall_score(true_labels_rou, pred_labels_rou)

# 计算 F1 分数
#f1 = f1_score(true_labels_rou, pred_labels_rou)

# 计算准确率
accuracy = accuracy_score(true_labels_rou, pred_labels_rou)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score


f1 = f1_score(true_labels, pred_labels, average='weighted')
# 计算精确度
precision = precision_score(true_labels, pred_labels)

# 计算召回率
recall = recall_score(true_labels, pred_labels)

# 计算 F1 分数
f1 = f1_score(true_labels, pred_labels)

# 计算准确率
accuracy = accuracy_score(true_labels, pred_labels)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Accuracy:", accuracy)